[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/53_rag_memory_solution.ipynb)

#  Solution: RAG Memory for Agents

Reference: Voyager (Wang et al. 2023), Generative Agents (Park et al. 2023), standard RAG retrieval.

```text
1. Embed query with embedding_fn
2. For each memory: cosine_sim = dot(q_emb, m_emb) / (|q|*|m|)
3. Keyword bonus: if query words appear in memory content, +0.1
4. Sort by score desc, return top_k
```

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
#  SOLUTION

def retrieve_relevant_memory(query, memory_store, embedding_fn, top_k=5):
    """Retrieve relevant past experiences.

    Inspired by:
    - Voyager (Wang et al. 2023): Minecraft agent with skill library + embedding retrieval
    - Generative Agents (Park et al. 2023): 25 agents with memory retrieval for social simulation
    - Standard RAG: encode → index → retrieve → rerank pipeline

    Cosine similarity is preferred over dot product because it normalizes
    for embedding magnitude differences (common across different text lengths).
    """
    query_emb = embedding_fn(query)

    scored = []
    for entry in memory_store:
        # Cosine similarity = dot(q, m) / (|q| * |m|)
        mem_emb = entry["embedding"]
        q_norm = query_emb.norm()
        m_norm = mem_emb.norm()
        if q_norm > 0 and m_norm > 0:
            cosine_sim = (query_emb @ mem_emb) / (q_norm * m_norm)
        else:
            cosine_sim = 0.0

        # Keyword match bonus: boost score if query words appear in content
        keyword_bonus = 0.0
        for word in query.lower().split():
            if word in entry["content"].lower():
                keyword_bonus = 0.1
                break

        scored.append((cosine_sim.item() + keyword_bonus, entry))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [entry for _, entry in scored[:top_k]]

In [ ]:
#  Verify
def emb(text):
    v = torch.tensor([len(text), float(ord(text[0])), float(len(text)*2)])
    return v / v.norm()
store = [
    {"embedding": emb("hello"), "content": "hello there", "metadata": {}},
    {"embedding": emb("world"), "content": "world here", "metadata": {}},
]
result = retrieve_relevant_memory("hello", store, emb, top_k=1)
print(f"Top: {result[0]['content']}")

In [ ]:
from torch_judge import check
check("rag_memory")